In [7]:
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Códigos')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install prefect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.5/213.5 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.9/348.9 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.

In [8]:
from prefect import flow, task
from google.colab import drive
import sys

import preprocessing as prep
import training as aml
import inference as inf
import posprocessing as posp

@task #jobs | contenido de los estados
def preprocess_data(payload):
  DIR_RAWDATA = payload['DIR_RAWDATA']
  DIR_PROCESSED = payload['DIR_PROCESSED']
  partition_id = payload['params']['partition']
  model_name = payload['params']['model_name']
  mode_type = payload['params']['mode_type']
  print("Running preprocessing")
  prep.main(model_name, DIR_RAWDATA, DIR_PROCESSED, mode_type, partition_id) #script mode
  dir_pre_processed = f'{DIR_PROCESSED}/preprocessed/vars_{partition_id}_{model_name}.csv'
  dir_pos_processed = f'{DIR_PROCESSED}/postprocessed/post_{partition_id}_{model_name}.csv'
  return dir_pre_processed, dir_pos_processed

@task
def train_model(payload):
  DATA_TRAIN_PATH = payload['TRAINING_DATA'] + '/train_vars_extrac.csv'
  DATA_TEST_PATH = payload['TRAINING_DATA'] + '/test_vars_extrac.csv'
  MODEL_DIR = payload['MODEL_DIR']
  print("Running training")
  aml.main(DATA_TRAIN_PATH, DATA_TEST_PATH, MODEL_DIR) #script mode

@task
def run_inference(payload, dir_pre_processed):
  SCORE_DIR = payload['SCORE_DIR']
  MODEL_DIR = payload['MODEL_DIR']
  partition_id = payload['params']['partition']
  model_name = payload['params']['model_name']
  print("Running inference")
  inf.main(MODEL_DIR, dir_pre_processed, SCORE_DIR) #script mode
  score_file_path = f'{SCORE_DIR}/inference_{model_name}_{partition_id}.csv'
  return score_file_path

@task
def postprocess_results(payload, dir_pos_processed, score_file_path):
  DIR_OUTPUT = payload['DIR_OUTPUT']
  partition_id = payload['params']['partition']
  print("Running postprocessing")
  posp.main(dir_pos_processed, score_file_path, DIR_OUTPUT) #script mode

@flow
def mlops_pipeline(payload): #inicializador
  # Preprocessing
  dir_pre_processed, dir_pos_processed = preprocess_data(payload)
  # Training - estado de elección (choose)
  if payload['params']['mode_type'] == 'training':
      train_model(payload) #script mode
  # Inference
  score_file_path = run_inference(payload, dir_pre_processed=dir_pre_processed)
  # Postprocessing for the specific partition
  final_output_path = postprocess_results(payload, dir_pos_processed=dir_pos_processed, score_file_path=score_file_path)

In [16]:
payload = {
        "DIR_RAWDATA": '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Cruda',
        "DIR_PROCESSED": '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Preprocesada',
        "MODEL_DIR": '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Best_model',
        "TRAINING_DATA": '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Data Preprocesada/training_data/preprocessed',
        "SCORE_DIR": '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Output/score',
        "DIR_OUTPUT": '/content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Output/final',
        "params": {
            "model_name": 'extrac', # para el guardado de la data
            "mode_type": 'inference',
            "partition": 8
        }
    }

In [17]:
mlops_pipeline(payload)

INFO:prefect.flow_runs:Beginning flow run 'hilarious-iguana' for flow 'mlops-pipeline'


Running preprocessing


INFO:prefect.task_runs:Finished in state Completed()


Running inference
Modelo: /content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Best_model/2026-05-06_02-07-17


INFO:prefect.task_runs:Finished in state Completed()


Predictions guardadas en: /content/drive/MyDrive/Docencia UCSP/Maestría DS 2024 - aplicaciones prácticas/Datos/Output/score/inference_extrac_8.csv
Running postprocessing


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.flow_runs:Finished in state Completed()
